In [35]:
library(lme4)
library(lmerTest)

In [36]:
df <- read.csv("concatenated_data/experiment3_acceptance_threshold.csv")
df$lang <- interaction(df$lang, df$bilingual)
df$lang <- as.character(df$lang)
df$lang[df$lang=="man.False"] <- "man"
df$lang[df$lang=="man.True"] <- "man_bilingual"
df$lang[df$lang=="min.True"] <- "min"

df[df$tone=="tone21",]$acceptance_threshold <- (df[df$tone=="tone21",]$acceptance_threshold-11)*-1 # flip the scale for tone21 for comparison
df$tone <- as.factor(df$tone)
contrasts(df$tone) <- c(-.5, .5, levels=c("tone21", "tone51"))
df$lang <- as.factor(df$lang)
print(contrasts(df$lang))
df$participant <- as.factor(df$participant)
df$bilingual <- as.factor(df$bilingual)
contrasts(df$bilingual) <- c(-.5, .5, levels=c("False", "True"))
df$strict <- as.factor(df$strict)
df$advanced <- as.factor(df$advanced)

              man_bilingual min
man                       0   0
man_bilingual             1   0
min                       0   1


In [37]:
# man_monolingual vs. man_bilingual vs. min vs. two tones
temp <- df
temp$lang <- droplevels(temp$lang)
temp$tone <- droplevels(temp$tone)

temp$tone <- factor(temp$tone, levels = c("tone21", "tone51"))
contrasts(temp$tone) <- c(-0.5, 0.5)
print(contrasts(temp$tone))
temp$lang <- as.factor(temp$lang)
print(contrasts(temp$lang))

model <- lmer(acceptance_threshold ~ lang*tone + (1|participant), data=temp)
summary(model)

       [,1]
tone21 -0.5
tone51  0.5
              man_bilingual min
man                       0   0
man_bilingual             1   0
min                       0   1


Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: acceptance_threshold ~ lang * tone + (1 | participant)
   Data: temp

REML criterion at convergence: 362.7

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-4.2795 -0.3453 -0.0565  0.7826  1.6680 

Random effects:
 Groups      Name        Variance Std.Dev.
 participant (Intercept) 0.0768   0.2771  
 Residual                0.6669   0.8167  
Number of obs: 142, groups:  participant, 43

Fixed effects:
                        Estimate Std. Error       df t value Pr(>|t|)    
(Intercept)               5.1333     0.1654  86.9556  31.039  < 2e-16 ***
langman_bilingual        -0.1333     0.2049  86.9556  -0.651  0.51704    
langmin                  -0.3833     0.2049  86.9556  -1.870  0.06480 .  
tone1                     2.0000     0.2982 101.4249   6.707 1.15e-09 ***
langman_bilingual:tone1  -0.1429     0.3695 101.4249  -0.387  0.69988    
langmin:tone1             1.1429     0.3695 1

In [38]:
library(emmeans)

emm <- emmeans(
  model,
  ~ lang * tone,
  weights = "flat"
)

summary(emm)

tone_within_lang <- emmeans(
  model,
  ~ lang | tone,
  weights = "flat"
)

pairs(
  tone_within_lang
)

,lang,tone,emmean,SE,df,lower.CL,upper.CL
,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,man,tone21,4.133333,0.2226723,132.3636,3.692877,4.573790
2,man_bilingual,tone21,4.071429,0.1629794,132.3636,3.749047,4.393810
3,min,tone21,3.178571,0.1629794,132.3636,2.856190,3.500953
4,man,tone51,6.133333,0.2226723,132.3636,5.692877,6.573790
5,man_bilingual,tone51,5.928571,0.1629794,132.3636,5.606190,6.250953
6,min,tone51,6.321429,0.1629794,132.3636,5.999047,6.643810


tone = tone21:
 contrast            estimate    SE  df t.ratio p.value
 man - man_bilingual   0.0619 0.276 132   0.224  0.9726
 man - min             0.9548 0.276 132   3.460  0.0021
 man_bilingual - min   0.8929 0.218  97   4.091  0.0003

tone = tone51:
 contrast            estimate    SE  df t.ratio p.value
 man - man_bilingual   0.2048 0.276 132   0.742  0.7389
 man - min            -0.1881 0.276 132  -0.682  0.7746
 man_bilingual - min  -0.3929 0.218  97  -1.800  0.1750

Degrees-of-freedom method: kenward-roger 
P value adjustment: tukey method for comparing a family of 3 estimates 

In [39]:
# man_monolingual vs. man_bilingual vs. min for tone21
temp <- df
temp <- temp[(temp$tone=="tone21"),]
temp$lang <- droplevels(temp$lang)
temp$tone <- droplevels(temp$tone)
model <- lmer(acceptance_threshold ~ lang + (1|participant), data=temp)
summary(model)

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: acceptance_threshold ~ lang + (1 | participant)
   Data: temp

REML criterion at convergence: 169.1

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-2.7774 -0.2215 -0.1240  1.0148  1.2580 

Random effects:
 Groups      Name        Variance Std.Dev.
 participant (Intercept) 0.02031  0.1425  
 Residual                0.59284  0.7700  
Number of obs: 71, groups:  participant, 43

Fixed effects:
                  Estimate Std. Error      df t value Pr(>|t|)    
(Intercept)         4.1333     0.2022 67.9495  20.444  < 2e-16 ***
langman_bilingual  -0.0619     0.2505 67.9495  -0.247 0.805595    
langmin            -0.9548     0.2505 67.9495  -3.811 0.000301 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Correlation of Fixed Effects:
            (Intr) lngmn_
lngmn_blngl -0.807       
langmin     -0.807  0.663

In [40]:
# man_monolingual vs. min for tone51
temp <- df
temp <- temp[(temp$tone=="tone51"),]
temp$lang <- droplevels(temp$lang)
temp$tone <- droplevels(temp$tone)

model <- lm(acceptance_threshold ~ lang, data=temp)
summary(model)


Call:
lm(formula = acceptance_threshold ~ lang, data = temp)

Residuals:
    Min      1Q  Median      3Q     Max 
-3.9286 -0.3214  0.0714  0.6786  1.0714 

Coefficients:
                  Estimate Std. Error t value Pr(>|t|)    
(Intercept)         6.1333     0.2419  25.352   <2e-16 ***
langman_bilingual  -0.2048     0.2998  -0.683    0.497    
langmin             0.1881     0.2998   0.627    0.532    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.937 on 68 degrees of freedom
Multiple R-squared:  0.03494,	Adjusted R-squared:  0.006559 
F-statistic: 1.231 on 2 and 68 DF,  p-value: 0.2984


In [41]:
# man_bilingual vs. min for tone21
temp <- df
temp <- temp[(temp$tone=="tone21"),]
temp <- temp[temp$lang!="man",]
temp$lang <- droplevels(temp$lang)
temp$tone <- droplevels(temp$tone)

model <- lmer(acceptance_threshold ~ lang + (1|participant), data=temp)
summary(model)

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: acceptance_threshold ~ lang + (1 | participant)
   Data: temp

REML criterion at convergence: 137.9

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.66472 -0.21206 -0.07837  0.96042  1.21257 

Random effects:
 Groups      Name        Variance Std.Dev.
 participant (Intercept) 0.02381  0.1543  
 Residual                0.64220  0.8014  
Number of obs: 56, groups:  participant, 28

Fixed effects:
            Estimate Std. Error      df t value Pr(>|t|)    
(Intercept)   4.0714     0.1542 53.9311  26.399  < 2e-16 ***
langmin      -0.8929     0.2142 27.0000  -4.169 0.000283 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Correlation of Fixed Effects:
        (Intr)
langmin -0.694

In [42]:
# man_bilingual vs. min for tone51
temp <- df
temp <- temp[(temp$tone=="tone51"),]
temp <- temp[temp$lang!="man",]
temp$lang <- droplevels(temp$lang)
temp$tone <- droplevels(temp$tone)

model <- lmer(acceptance_threshold ~ lang + (1|participant), data=temp)
summary(model)

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: acceptance_threshold ~ lang + (1 | participant)
   Data: temp

REML criterion at convergence: 136.1

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.51684 -0.52349 -0.03434  0.64575  1.54028 

Random effects:
 Groups      Name        Variance Std.Dev.
 participant (Intercept) 0.7646   0.8744  
 Residual                0.2348   0.4845  
Number of obs: 56, groups:  participant, 28

Fixed effects:
            Estimate Std. Error      df t value Pr(>|t|)    
(Intercept)   5.9286     0.1889 34.0627  31.381  < 2e-16 ***
langmin       0.3929     0.1295 27.0000   3.034  0.00529 ** 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Correlation of Fixed Effects:
        (Intr)
langmin -0.343